<a href="https://colab.research.google.com/github/why2011btv/quant/blob/main/xtx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To open a CSV file and view its first few rows, you'll typically use the `pandas` library. First, you'll need to import `pandas` and then use `pd.read_csv()` to load your data. Finally, `df.head()` will show you the beginning of your DataFrame.

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the actual path to your CSV file
try:
    df = pd.read_csv('/content/drive/MyDrive/XTX/train/stock/AAL.csv')
    print("Successfully loaded 'AAL.csv'. Here are the first 5 rows:")
    display(df.head())
except FileNotFoundError:
    print("Error: 'AAL.csv' not found. Please make sure the file path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 'AAL.csv'. Here are the first 5 rows:


,Date,y_1,y_2
0,2013-01-02,-0.023156,0.059698
1,2013-01-03,0.075414,0.104188
2,2013-01-04,0.007440,0.026793
3,2013-01-07,0.021334,0.020014
4,2013-01-08,-0.001981,-0.025386


To mount your Google Drive, run the following code. You'll be prompted to authorize access to your Google Drive account.

Once mounted, you can verify it by listing the contents of your Google Drive. For example, to see what's directly in 'My Drive':

In [ ]:
!ls /content/drive/MyDrive/XTX/train/stock

AAL.csv   BABA.csv  CMS.csv   EL.csv	GSK.csv   NRG.csv   RRC.csv   UNP.csv
AAP.csv   BAM.csv   CNP.csv   ENB.csv	HA.csv	  NTAP.csv  RS.csv    UPS.csv
ABBV.csv  BBY.csv   COF.csv   EPD.csv	HBAN.csv  NTES.csv  RSG.csv   URBN.csv
ACN.csv   BEN.csv   COP.csv   EQT.csv	H.csv	  NUE.csv   RY.csv    URI.csv
ADI.csv   BIDU.csv  COST.csv  ERIC.csv	HES.csv   NVAX.csv  SPLK.csv  USB.csv
ADSK.csv  BIG.csv   COTY.csv  ETN.csv	HIG.csv   NVO.csv   SPR.csv   USO.csv
AEO.csv   BIIB.csv  CPB.csv   ETR.csv	HLF.csv   NVR.csv   SPWR.csv  VALE.csv
AEP.csv   BIP.csv   CP.csv    EXC.csv	HOG.csv   NWL.csv   SU.csv    V.csv
AES.csv   BK.csv    CRM.csv   EXEL.csv	HP.csv	  NYCB.csv  SWK.csv   VEEV.csv
AFL.csv   BMRN.csv  CSIQ.csv  EXPE.csv	HRL.csv   OC.csv    SWKS.csv  VFC.csv
AGNC.csv  BRO.csv   CSX.csv   FANG.csv	HSBC.csv  OKE.csv   SWN.csv   VLO.csv
AI.csv	  BSX.csv   CTSH.csv  FDX.csv	HSIC.csv  ORCL.csv  SYY.csv   VRTX.csv
AIG.csv   BTU.csv   CUK.csv   FITB.csv	HUM.csv   ORLY.csv  TCOM.csv  W.csv
AKAM.csv  BX.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:

import pandas as pd
import numpy as np
import os
import glob
import torch
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb

# --- 配置区 ---
#
PROJ_DIR = '/content/drive/MyDrive/XTX'
DATA_DIR = PROJ_DIR + '/train'
DEBUG_N_ROWS = None  # 【关键】只读取前 100 条新闻用于测试
BATCH_SIZE = 64      # CPU 上 batch_size 调小一点
CACHE_PATH = PROJ_DIR + '/train/news_with_sentiment.csv'  # 缓存文件的名字

# --- 1. NLP 特征工程 (简化版) ---
class SentimentAnalyzerLite:
    def __init__(self, model_name="ProsusAI/finbert"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"🚀 Device set to: {self.device}")

        # 打印一下具体显卡名字，确保你真的连上了 T4
        if self.device.type == 'cuda':
            print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
        else:
            print("⚠️ Warning: Running on CPU! This will be slow.")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # 这里的模型加载会消耗一些内存，但只跑一次
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)

    def get_sentiment(self, texts):
        self.model.eval()
        all_probs = []

        # 在 CPU 上，每一步都打印进度
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="FinBERT Processing"):
            batch_texts = texts[i:i + BATCH_SIZE]
            inputs = self.tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=64)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                #all_probs.append(probs.numpy()) # CPU Tensor直接转numpy
                all_probs.append(probs.cpu().numpy())

        return np.concatenate(all_probs, axis=0)

def get_news_with_sentiment(news_path, debug_n_rows=None):
    # 修改缓存文件名，避免和只读 Title 的旧缓存混淆
    cache_name = 'news_with_sentiment_full_text.csv'
    if debug_n_rows:
        cache_name = f'news_with_sentiment_full_text_{debug_n_rows}.csv'

    current_cache_path = os.path.join(PROJ_DIR, 'train', cache_name)

    if os.path.exists(current_cache_path):
        print(f"✅ 发现包含 Summary 的缓存: {current_cache_path}，直接读取...")
        return pd.read_csv(current_cache_path)

    print(f"⚠️ 未发现缓存，开始运行 FinBERT (包含 Title + Summary)...")

    # 读取原始 CSV
    news = pd.read_csv(news_path, nrows=debug_n_rows)

    # 【关键修改】文本拼接
    # 1. 填充空值
    news['Title'] = news['Title'].fillna('')
    news['Summary_Textrank'] = news['Summary_Textrank'].fillna('')

    # 2. 拼接：标题 + 句号 + 摘要
    # 这样 FinBERT 就能读到完整的故事了
    print("正在拼接 Title 和 Summary...")
    news['combined_text'] = news['Title'] + ". " + news['Summary_Textrank']

    # 3. 截断保护 (虽然 Tokenizer 会做，但我们先简单处理一下过长的文本)
    # FinBERT max_length 通常是 512 tokens。Summary 可能会很长。
    # 这里我们不做硬截断，交给 Tokenizer 的 truncation=True 处理即可。

    # 运行 NLP
    analyzer = SentimentAnalyzerLite()
    print("正在计算 Combined Sentiment...")

    # 喂给模型的是 combined_text
    sentiment_scores = analyzer.get_sentiment(news['combined_text'].tolist())
    sentiment_df = pd.DataFrame(sentiment_scores, columns=['pos', 'neg', 'neu'])

    # 合并
    news_processed = pd.concat([news, sentiment_df], axis=1)

    # 保存缓存
    print(f"💾 保存新缓存到 {current_cache_path} ...")
    news_processed.to_csv(current_cache_path, index=False)

    return news_processed


def calculate_ema(x, span):
    # adjust=False 意味着我们使用递归公式: y_t = alpha * x_t + (1 - alpha) * y_{t-1}
    # 这是金融领域最常用的标准
    return x.ewm(span=span, adjust=False).mean()


# 定义标准差计算 (衡量情绪波动)
def calculate_std(x, window):
    return x.rolling(window=window, min_periods=1).std()


def load_data_lite():
    # 1. 加载 News
    # 注意：这里我们让它强制转成 UTC 再去除时区，确保万无一失
    news = get_news_with_sentiment(PROJ_DIR + '/train/news.csv', debug_n_rows=DEBUG_N_ROWS)

    # 【关键修复 A】处理 News 日期：统一转成 UTC -> 然后剥离时区
    news['Date'] = pd.to_datetime(news['Date'], utc=True, errors='coerce').dt.tz_localize(None)

    # ... (聚合逻辑不变) ...
    print("正在聚合数据 (Aggregation)...")
    daily_features = news.groupby(['Date', 'Symbol']).agg({
        'pos': 'mean', 'neg': 'mean', 'neu': 'mean', 'Title': 'count'
    }).rename(columns={'Title': 'volume'})

    daily_features = daily_features.reset_index().sort_values(['Symbol', 'Date'])

    # ... (EMA 计算逻辑不变) ...
    grouped = daily_features.groupby('Symbol')
    daily_features['pos_ema_short'] = grouped['pos'].transform(lambda x: calculate_ema(x, span=3))
    daily_features['neg_ema_short'] = grouped['neg'].transform(lambda x: calculate_ema(x, span=3))
    daily_features['pos_ema_mid'] = grouped['pos'].transform(lambda x: calculate_ema(x, span=10))
    daily_features['neg_ema_mid'] = grouped['neg'].transform(lambda x: calculate_ema(x, span=10))
    daily_features['vol_ema_short'] = grouped['volume'].transform(lambda x: calculate_ema(x, span=3))
    # --- 新增：情绪波动率特征 ---
    # 逻辑: 过去 5 天，这只股票的消息面是否“精神分裂”？
    # 波动率大往往意味着变盘点
    daily_features['pos_std_5d'] = grouped['pos'].transform(lambda x: calculate_std(x, 5))
    daily_features['neg_std_5d'] = grouped['neg'].transform(lambda x: calculate_std(x, 5))

    # 还可以加个 Volume 的波动率 (忽大忽小的关注度也是信号)
    daily_features['vol_std_5d'] = grouped['volume'].transform(lambda x: calculate_std(x, 5))
    # ... (在 calculate_std 之后) ...

    # === 新增：Lag Features (滞后特征) ===
    # 逻辑：昨天的情绪 (Lag 1) 和前天的情绪 (Lag 2) 往往能预测今天的走势
    # 尤其是 y_1 (短期回报)，往往是对昨天新闻的"补涨"或"补跌"

    # 必须先按 Symbol 分组，否则 shift 会把 A 股票的数据移给 B 股票
    daily_features['pos_lag1'] = grouped['pos'].shift(1)
    daily_features['neg_lag1'] = grouped['neg'].shift(1)
    daily_features['vol_lag1'] = grouped['volume'].shift(1)

    # 也可以加个 Lag 2
    daily_features['pos_lag2'] = grouped['pos'].shift(2)
    daily_features['neg_lag2'] = grouped['neg'].shift(2)
    daily_features['vol_lag2'] = grouped['volume'].shift(2)


    print("特征工程完成：已加入 Lag (滞后) 特征")
    # 记得 fillna，因为 std 计算可能会产生 NaN
    daily_features = daily_features.fillna(0)

    # 2. 加载 Stock
    print(f"--- 步骤 2: 动态加载相关 Stock 数据 ---")
    target_symbols = daily_features['Symbol'].unique()

    stock_dfs = []
    for symbol in target_symbols:
        path = os.path.join(DATA_DIR, 'stock', f'{symbol}.csv')
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['Symbol'] = symbol
            stock_dfs.append(df)

    if not stock_dfs:
        raise ValueError("没有找到对应的股票数据！")

    stocks = pd.concat(stock_dfs)

    # 【关键修复 B】处理 Stock 日期：同样统一转成 UTC -> 然后剥离时区
    stocks['Date'] = pd.to_datetime(stocks['Date'], utc=True, errors='coerce').dt.tz_localize(None)

    # --- 调试打印 ---
    # 现在打印出来应该都是 datetime64[ns]，没有 UTC 字样了
    print(f"News Date Type: {daily_features['Date'].dtype}")
    print(f"Stock Date Type: {stocks['Date'].dtype}")

    # 3. 合并
    full_df = pd.merge(stocks, daily_features, on=['Date', 'Symbol'], how='left')

    # 检查 Merge 质量
    missing_ratio = full_df['pos'].isna().mean()
    print(f"Merge 后缺失值比例: {missing_ratio:.2%} (正常应该是 80%-90% 左右，绝不能是 100%)")

    full_df = full_df.fillna(0)

    # 4. 交互特征
    print("正在构建交互特征...")
    full_df['interaction_pos_vol'] = full_df['pos_ema_short'] * full_df['vol_ema_short']
    full_df['interaction_neg_vol'] = full_df['neg_ema_short'] * full_df['vol_ema_short']
    full_df['sentiment_gap'] = full_df['pos_ema_short'] - full_df['neg_ema_short']
    full_df['interaction_net_vol'] = full_df['sentiment_gap'] * full_df['vol_ema_short']

    return full_df


def custom_r2_metric(y_true, y_pred):
    """
    题目要求的特殊 R2 计算公式。
    """
    # 转换为 numpy 数组以防万一
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    numerator = np.sum((y_pred - y_true)**2)
    denominator = np.sum(y_true**2)

    # 避免分母为0的极个别情况
    if denominator == 0:
        return 0.0

    return 1 - (numerator / denominator)

# --- 2. 训练与测试 ---
def run_pipeline_lite():
    # 1. 加载数据
    df = load_data_lite()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)

    exclude_cols = ['Date', 'Symbol', 'y_1', 'y_2']
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    # 【修复 Warning】不再转换成 numpy array (.values)，保留 DataFrame 格式
    X = df[feature_cols]

    targets = ['y_1', 'y_2']

    # 保存最后一个模型用于分析
    last_model = None

    for target in targets:
        print(f"\n=== 正在评估目标: {target} (LightGBM) ===")
        y = df[target] # Series 格式

        # CV
        tscv = TimeSeriesSplit(n_splits=3)
        fold_scores = []

        for fold_idx, (train_index, test_index) in enumerate(tscv.split(X)):
            # pandas 切片
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            # ... 在 X_train, X_test 切分之后 ...

            # --- DEBUG: 检查特征是否全是 0 ---
            if fold_idx == 0: # 只在第一折检查
                print("\n[Data Debugging]")
                print("训练集大小:", X_train.shape)
                print("特征方差 (Variance):")
                # 如果方差接近 0，说明这个特征几乎是常数，模型学不到东西
                print(X_train.var().sort_values(ascending=False).head(5))
                print("----------------")

            # --- 模型调优: 稳健版 (Conservative) ---
            model = lgb.LGBMRegressor(
                n_estimators=500,        # 树多一点
                learning_rate=0.01,      # 【关键】学习率调小，让它学得更细
                max_depth=5,             # 深度回调一点，防止过拟合
                num_leaves=20,           # 叶子少一点，提高泛化能力
                min_child_samples=50,    # 【关键】提高分裂门槛，过滤掉个股的噪音
                reg_alpha=0.1,           # L1 正则化 (新增)
                reg_lambda=1.0,          # L2 正则化 (新增，惩罚大数值预测)
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            )

            model.fit(X_train, y_train)
            pred_test = model.predict(X_test)

            score = custom_r2_metric(y_test, pred_test)
            fold_scores.append(score)
            print(f"  Fold {fold_idx+1}: R2: {score:.4f}")

            # 获取 Gain 重要性
            imp_values = model.booster_.feature_importance(importance_type='gain')
            last_importance_df = pd.DataFrame({
                'Feature': feature_cols,
                'Importance': imp_values
            }).sort_values('Importance', ascending=False)

            last_model = model # 暂存模型

        print(f"--- {target} 平均 R2: {np.mean(fold_scores):.4f} ---")

        if last_importance_df is not None:
            print(f"\n[{target} 特征贡献度 (Gain) Top 5]")
            print(last_importance_df.head(5))

            # 如果 Top 1 还是 0，那就强制打印所有非零的
            if last_importance_df['Importance'].iloc[0] == 0:
                print("⚠️ 警告: 所有特征贡献度均为 0！模型可能未进行任何分裂。")
            else:
                # 简单画个图
                print(f"有效特征数量: {(last_importance_df['Importance'] > 0).sum()}")

if __name__ == "__main__":
    # 确保路径存在，防止报错
    if not os.path.exists(PROJ_DIR + '/train/news.csv'):
        print("错误: 请确保当前目录下有 train/news.csv")
    else:
        run_pipeline_lite()

✅ 发现包含 Summary 的缓存: /content/drive/MyDrive/XTX/train/news_with_sentiment_full_text.csv，直接读取...
正在聚合数据 (Aggregation)...
特征工程完成：已加入 Lag (滞后) 特征
--- 步骤 2: 动态加载相关 Stock 数据 ---
News Date Type: datetime64[ns]
Stock Date Type: datetime64[ns]
Merge 后缺失值比例: 63.54% (正常应该是 80%-90% 左右，绝不能是 100%)
正在构建交互特征...

=== 正在评估目标: y_1 (LightGBM) ===

[Data Debugging]
训练集大小: (126997, 22)
特征方差 (Variance):
volume           1.023437
vol_lag1         1.006612
vol_lag2         1.001142
vol_ema_short    0.744348
vol_std_5d       0.365930
dtype: float64
----------------
  Fold 1: R2: -0.0016
  Fold 2: R2: -0.0011
  Fold 3: R2: -0.0002
--- y_1 平均 R2: -0.0010 ---

[y_1 特征贡献度 (Gain) Top 5]
       Feature  Importance
15    pos_lag2    4.505971
16    neg_lag2    3.184659
12    pos_lag1    2.332228
9   pos_std_5d    2.285960
1          neg    2.200175
有效特征数量: 22

=== 正在评估目标: y_2 (LightGBM) ===

[Data Debugging]
训练集大小: (126997, 22)
特征方差 (Variance):
volume           1.023437
vol_lag1         1.006612
vol_lag2         1.0011